# 19 — Reserving a Representative Threshold Sample

Notebook 18 selected a buffer by coverage and then split it, 70% to fit the model and 30% to choose the operating threshold. That removes the reuse of training rows but leaves a second problem: the threshold sample is itself coverage-selected, so it is not representative of ordinary target benign traffic. Residual false-positive violations could therefore come from selection bias rather than from small-sample estimation error, and the two have not been separated.

This notebook allocates the sampling roles before acquisition instead of after. A fixed share of the budget is drawn uniformly at random from the unlabelled pool and reserved for thresholding, and the remainder is acquired by coverage from the pool with those rows removed. The number of purchased labels is unchanged, so the comparison is at equal budget; what changes is whether the threshold sample was selected.

Four conditions run at each budget. **split_diversity** and **split_uniform** reproduce the notebook-18 design, splitting a buffer drawn by one rule. **reserved_diversity** and **reserved_uniform** reserve a representative threshold sample first. Comparing the first pair with the second isolates selection bias; comparing across budgets isolates sample size.

Per-cell label accounting is recorded rather than averaged: the benign count in the threshold sample, the false positives actually observed there at the chosen cut, and the resulting Clopper-Pearson upper bound. A bound computed from a mean sample size is not a bound for any particular detector, so these are reported per procedure.

Results append to fc_results_v9.csv.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, gc, time, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

BASE   = '/content/drive/MyDrive/drift-conference'
CACHE  = f'{BASE}/data/nfv2/cache'
RESULT = f'{BASE}/results/fourcorpus'

CFG = dict(corpora=['nf2018v2','nfunswv2','nftonv2','nfbotv2'], seed=42, test_size=0.30,
           eval_cap=200_000, pool_cap=200_000, budgets=[0.0001, 0.001],
           thr_frac=0.30, fpr_cap=0.01, ece_bins=15,
           rf_estimators=300, mlp_hidden=(128,64), mlp_max_iter=100)
MODELS = ['rf', 'lgbm', 'mlp']
V9_CSV = f'{RESULT}/fc_results_v9.csv'
COLS = ['seed','target','model','budget','design','rule','n_labels','n_fit','n_threshold',
        'n_benign_fit','n_benign_threshold','fp_observed_threshold','fp_bound_threshold',
        'benign_rate_pool','benign_rate_threshold',
        'mcc_at_threshold','tpr_dep','fpr_dep','tpr_oracle','thr','fit_s']
print(json.dumps({k: str(v) for k, v in CFG.items()}, indent=2))

In [ ]:
DATASETS = {tag: pd.read_parquet(f'{CACHE}/{tag}_prepared.parquet') for tag in CFG['corpora']}
FEATURES = [c for c in DATASETS['nf2018v2'].columns if c not in ('Label','Attack')]
print('features:', len(FEATURES))

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import (f1_score, matthews_corrcoef, average_precision_score,
                              brier_score_loss, confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

F32_SAFE = 1e37

def ece_score(y_true, p_pos, n_bins=15):
    conf = np.maximum(p_pos, 1 - p_pos)
    correct = ((p_pos >= 0.5).astype(int) == y_true).astype(float)
    bins = np.linspace(0.5, 1.0, n_bins + 1)
    idx = np.clip(np.digitize(conf, bins) - 1, 0, n_bins - 1)
    ece = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return ece

def mcc_from_counts(tp, fp, fn, tn):
    num = tp * tn - fp * fn
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    with np.errstate(invalid='ignore', divide='ignore'):
        out = np.where(den > 0, num / den, 0.0)
    return out

def best_threshold_mcc(y_true, p_pos, n_grid=199):
    """Max MCC over a quantile grid of thresholds; robust to degenerate score distributions."""
    y = np.asarray(y_true).astype(int); p = np.asarray(p_pos)
    thr = np.unique(np.quantile(p, np.linspace(0.001, 0.999, n_grid)))
    if len(thr) < 2:
        thr = np.array([thr[0]]) if len(thr) else np.array([0.5])
    P = y.sum(); N = len(y) - P
    tp = np.array([(y[p >= t]).sum() for t in thr], dtype=float)
    fp = np.array([(p >= t).sum() for t in thr], dtype=float) - tp
    fn = P - tp; tn = N - fp
    mccs = mcc_from_counts(tp, fp, fn, tn)
    i = int(np.argmax(mccs))
    return float(mccs[i]), float(thr[i])

def all_metrics(y_true, p_pos, ece_bins=CFG['ece_bins']):
    y_true = np.asarray(y_true); p_pos = np.asarray(p_pos)
    pred = (p_pos >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    ap_att = average_precision_score(y_true, p_pos)
    ap_ben = average_precision_score(1 - y_true, 1 - p_pos)
    mb, tb = best_threshold_mcc(y_true, p_pos)
    return dict(
        macro_f1=f1_score(y_true, pred, average='macro'),
        weighted_f1=f1_score(y_true, pred, average='weighted'),
        mcc=matthews_corrcoef(y_true, pred),
        auprc_macro=(ap_att + ap_ben) / 2,
        fp_rate=fp / (fp + tn) if (fp + tn) else np.nan,
        brier=brier_score_loss(y_true, p_pos),
        ece=ece_score(y_true, p_pos, ece_bins),
        mcc_best_thr=mb, thr_best=tb,
    )

def clean_X(df, features, medians=None):
    X = df[features].astype('float64')
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.mask(X.abs() > F32_SAFE, np.nan)
    if medians is None:
        medians = X.median()
    return X.fillna(medians), medians

def make_model(name, seed, n_rows, rf_estimators=CFG['rf_estimators'], mlp_hidden=CFG['mlp_hidden'], mlp_max_iter=CFG['mlp_max_iter']):
    if name == 'rf':
        return RandomForestClassifier(n_estimators=rf_estimators, n_jobs=-1, random_state=seed)
    if name == 'lgbm':
        return lgb.LGBMClassifier(n_estimators=rf_estimators, random_state=seed, n_jobs=-1, verbosity=-1)
    # early stopping needs a stratifiable validation split; disable on tiny buffers
    small = n_rows < 5000
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf', MLPClassifier(hidden_layer_sizes=mlp_hidden, activation='relu', solver='adam',
                              batch_size=min(1024, max(8, n_rows // 4)),
                              max_iter=(200 if small else mlp_max_iter),
                              early_stopping=not small, validation_fraction=0.05,
                              n_iter_no_change=5, random_state=seed)),
    ])

def platt_fit(scores, y):
    y = np.asarray(y)
    if len(np.unique(y)) < 2:
        return None
    lr = LogisticRegression(max_iter=1000)
    lr.fit(np.asarray(scores).reshape(-1, 1), y)
    return lr

def platt_apply(lr, scores, y_buf):
    if lr is None:
        return np.full(len(scores), float(np.asarray(y_buf)[0]))
    return lr.predict_proba(np.asarray(scores).reshape(-1, 1))[:, 1]

def stratified_frac(df, frac, seed, min_per_group=1):
    """Per-family sample of round(n*frac) rows, floored at min_per_group; no groupby.apply."""
    rng_state = seed
    parts = []
    for fam, g in df.groupby('Attack', sort=True):
        n = min(len(g), max(min_per_group, int(round(len(g) * frac))))
        parts.append(g.sample(n=n, random_state=rng_state))
    return pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)

def stratified_cap(df, cap, seed):
    if len(df) <= cap:
        return df.reset_index(drop=True)
    return stratified_frac(df, cap / len(df), seed)

In [ ]:
import numpy as np, pandas as pd
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import matthews_corrcoef

def mcc_from_counts(tp, fp, fn, tn):
    num = tp * tn - fp * fn
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    with np.errstate(invalid='ignore', divide='ignore'):
        return np.where(den > 0, num / den, 0.0)

def best_threshold_mcc(y_true, p_pos):
    y = np.asarray(y_true).astype(np.int64); p = np.asarray(p_pos, dtype=np.float64)
    order = np.argsort(-p, kind='mergesort'); ps, ys = p[order], y[order]
    P = int(ys.sum()); N = len(ys) - P
    tp = np.cumsum(ys); fp = np.cumsum(1 - ys)
    last = np.r_[ps[1:] != ps[:-1], True]
    tp, fp, cuts = tp[last].astype(float), fp[last].astype(float), ps[last]
    mccs = mcc_from_counts(tp, fp, P - tp, N - fp)
    i = int(np.argmax(mccs))
    return (0.0, float('inf')) if mccs[i] <= 0 else (float(mccs[i]), float(cuts[i]))

def best_threshold_two_sided(y_true, p_pos):
    """Max MCC over both orientations. Returns (mcc, thr, orient) with orient +1 (p>=thr) or -1 ((1-p)>=thr)."""
    m_pos, t_pos = best_threshold_mcc(y_true, p_pos)
    m_neg, t_neg = best_threshold_mcc(y_true, 1.0 - np.asarray(p_pos))
    return (m_pos, t_pos, 1) if m_pos >= m_neg else (m_neg, t_neg, -1)

def apply_oriented(p_pos, thr, orient):
    p = np.asarray(p_pos)
    return ((p if orient == 1 else 1.0 - p) >= thr).astype(int)

def mcc_of_pred(y_true, pred):
    return matthews_corrcoef(np.asarray(y_true), np.asarray(pred))

# ---------- acquisition rules: return integer positions into the pool ----------
def acquire_uniform(n_pool, k, seed):
    return np.random.default_rng(seed).choice(n_pool, size=min(k, n_pool), replace=False)

def acquire_uncertainty(p_pool, k):
    p = np.clip(np.asarray(p_pool), 1e-9, 1 - 1e-9)
    ent = -(p * np.log(p) + (1 - p) * np.log(1 - p))
    return np.argsort(-ent, kind='mergesort')[:k]

def acquire_diversity(X_pool, k, seed):
    """k-means with k clusters on standardised features; per cluster, the member nearest its centroid.
    Distances are computed to each row's own centroid only (O(n*features)), never as an n x k matrix."""
    Xs = StandardScaler().fit_transform(np.asarray(X_pool, dtype=np.float64))
    k = min(k, len(Xs))
    km = MiniBatchKMeans(n_clusters=k, random_state=seed, batch_size=4096, n_init=1, max_iter=50).fit(Xs)
    labels = km.labels_
    own = np.einsum('ij,ij->i', Xs - km.cluster_centers_[labels], Xs - km.cluster_centers_[labels])
    df = pd.DataFrame({'lab': labels, 'd': own})
    chosen = df.groupby('lab').d.idxmin().values.astype(int)
    if len(chosen) < k:
        rest = np.setdiff1d(np.arange(len(Xs)), chosen)
        chosen = np.concatenate([chosen, rest[np.argsort(own[rest])[:k - len(chosen)]]])
    return chosen

def acquire_hybrid(X_pool, p_pool, k, seed, factor=5):
    cand = acquire_uncertainty(p_pool, min(len(p_pool), factor * k))
    sub = acquire_diversity(np.asarray(X_pool)[cand], k, seed)
    return cand[sub]

# ---------- cross-validated buffer-only estimate on the buffer itself ----------
def cv_estimate(make_model_fn, Xb, yb, seed, n_splits=3):
    """Mean MCC over stratified folds; returns 0.0 when a class has fewer than n_splits rows."""
    yb = np.asarray(yb)
    if len(np.unique(yb)) < 2 or np.bincount(yb).min() < n_splits:
        return 0.0
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    out = []
    for tr, te in skf.split(Xb, yb):
        if len(np.unique(yb[tr])) < 2:
            out.append(0.0); continue
        mdl = make_model_fn(len(tr)); mdl.fit(Xb.iloc[tr], yb[tr])
        out.append(mcc_of_pred(yb[te], (mdl.predict_proba(Xb.iloc[te])[:, 1] >= 0.5).astype(int)))
    return float(np.mean(out))

In [ ]:
import numpy as np, pandas as pd

def tpr_at_fpr(y, p, target_fpr):
    """Highest TPR attainable at or below target_fpr, with the threshold that attains it.
    Thresholds are evaluated at every distinct score, so the result is exact."""
    y = np.asarray(y).astype(int); p = np.asarray(p, dtype=float)
    order = np.argsort(-p, kind='mergesort'); ys, ps = y[order], p[order]
    P = int(ys.sum()); N = len(ys) - P
    if P == 0 or N == 0:
        return np.nan, np.nan
    tp = np.cumsum(ys); fp = np.cumsum(1 - ys)
    last = np.r_[ps[1:] != ps[:-1], True]
    tp, fp, cuts = tp[last], fp[last], ps[last]
    ok = (fp / N) <= target_fpr
    if not ok.any():
        return 0.0, float(cuts[0]) + 1e-12
    i = int(np.argmax(np.where(ok, tp, -1)))
    return float(tp[i] / P), float(cuts[i])

def threshold_for_fpr_on_buffer(y_buf, p_buf, target_fpr):
    """Operating point an analyst could actually set: chosen on the labelled buffer only."""
    _, thr = tpr_at_fpr(y_buf, p_buf, target_fpr)
    return thr

def realised_at_threshold(y, p, thr):
    """TPR and FPR on the evaluation set at an externally chosen threshold."""
    y = np.asarray(y).astype(int); pred = (np.asarray(p) >= thr).astype(int)
    P = int(y.sum()); N = len(y) - P
    tp = int(((pred == 1) & (y == 1)).sum()); fp = int(((pred == 1) & (y == 0)).sum())
    return (tp / P if P else np.nan), (fp / N if N else np.nan)

def family_holdout_buffer(train_full, held_family, frac, seed, min_per_group=1):
    """Stratified buffer drawn only from families other than held_family, so the retrained
    model has never seen that attack type. Size matches the ordinary buffer at this budget."""
    pool = train_full[train_full['Attack'] != held_family]
    k = max(1, int(round(len(train_full) * frac)))
    parts = []
    for fam, g in pool.groupby('Attack', sort=True):
        n = min(len(g), max(min_per_group, int(round(len(g) * k / len(pool)))))
        parts.append(g.sample(n=n, random_state=seed))
    return pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)

def eligible_families(df, min_rows=2000, max_share=0.60):
    """Attack families large enough to matter but not so dominant that holding one out
    leaves nothing to train on."""
    v = df.loc[df['Attack'] != 'Benign', 'Attack'].value_counts()
    n_att = int((df['Attack'] != 'Benign').sum())
    return [f for f, c in v.items() if c >= min_rows and c / n_att <= max_share]

def metrics_at_threshold(y_true, p_pos, thr):
    """Threshold-sensitive metrics taken at an externally chosen cut rather than at 0.5.
    Needed because the rethreshold strategy does not operate at 0.5, so reporting its
    macro-F1 or false-positive rate from the default cut would describe a different detector."""
    from sklearn.metrics import f1_score, matthews_corrcoef, confusion_matrix
    y = np.asarray(y_true).astype(int)
    pred = (np.asarray(p_pos) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(mcc=matthews_corrcoef(y, pred),
                macro_f1=f1_score(y, pred, average='macro'),
                fp_rate=fp / (fp + tn) if (fp + tn) else np.nan)

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, matthews_corrcoef

def split_buffer(buf, thr_frac, seed, label_col='Label'):
    """Split a labelled buffer into a part used to fit the model and a part reserved for
    choosing the operating threshold. Both parts are drawn from the same budget, so the
    total label count is unchanged: thresholds are no longer selected on training rows."""
    rng = np.random.default_rng(seed)
    idx = np.arange(len(buf))
    thr_idx = []
    for _, g in buf.groupby(label_col, sort=True):
        pos = np.where(buf[label_col].values == g[label_col].iloc[0])[0]
        k = max(1, int(round(len(pos) * thr_frac))) if len(pos) > 1 else 0
        if k:
            thr_idx.extend(rng.choice(pos, size=min(k, len(pos) - 1), replace=False))
    thr_idx = np.array(sorted(set(thr_idx)), dtype=int)
    fit_idx = np.setdiff1d(idx, thr_idx)
    return buf.iloc[fit_idx], buf.iloc[thr_idx]

def orientation_signs(y, p):
    """The three quantities that must be distinguished: the sign of the score-label
    covariance, the sign of AUROC - 0.5, and which threshold orientation attains the higher
    MCC. They are not equivalent, so each is measured rather than inferred from another."""
    y = np.asarray(y).astype(int); p = np.asarray(p, dtype=float)
    # named cov_sy rather than cov: a column called 'cov' on a DataFrame is shadowed by
    # the DataFrame.cov method under attribute access, which silently yields the method
    out = dict(cov_sy=np.nan, auroc=np.nan, mcc_inc=np.nan, mcc_dec=np.nan)
    if len(np.unique(y)) < 2 or p.std() < 1e-12:
        return out
    out['cov_sy'] = float(np.cov(p, y, bias=True)[0, 1])
    out['auroc'] = float(roc_auc_score(y, p))
    order = np.argsort(-p, kind='mergesort'); ys = y[order]; ps = p[order]
    P = int(ys.sum()); N = len(ys) - P
    tp = np.cumsum(ys); fp = np.cumsum(1 - ys)
    last = np.r_[ps[1:] != ps[:-1], True]
    tp, fp = tp[last], fp[last]
    def mcc(tp, fp, fn, tn):
        d = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
        return np.where(d > 0, (tp * tn - fp * fn) / np.maximum(d, 1e-300), 0.0)
    out['mcc_inc'] = float(np.max(mcc(tp, fp, P - tp, N - fp)))
    tp2, fp2 = P - tp, N - fp
    out['mcc_dec'] = float(np.max(mcc(tp2, fp2, tp, fp)))
    return out

def tpr_at_common_fpr(y, p, cap):
    """Highest TPR attainable at or below a false-positive rate measured on the evaluation
    set itself. This is an oracle diagnostic, not a deployable threshold rule."""
    y = np.asarray(y).astype(int); p = np.asarray(p, dtype=float)
    P, N = int(y.sum()), int((1 - y).sum())
    if P == 0 or N == 0:
        return np.nan
    order = np.argsort(-p, kind='mergesort'); ys = y[order]; ps = p[order]
    tp = np.cumsum(ys); fp = np.cumsum(1 - ys)
    last = np.r_[ps[1:] != ps[:-1], True]
    tp, fp = tp[last], fp[last]
    ok = (fp / N) <= cap
    return float(tp[ok].max() / P) if ok.any() else 0.0

def uniform_buffer(train_full, frac, seed):
    """A label-blind sample: rows are drawn at random from the unlabelled pool with no use
    of family or class information, which is what an operator can actually do before paying
    for labels. The family-stratified buffer used elsewhere requires knowing attack families
    in advance and is therefore an oracle reference."""
    k = max(1, int(round(len(train_full) * frac)))
    return train_full.sample(n=min(k, len(train_full)), random_state=seed)

In [ ]:
import numpy as np

def label_counts(df, label_col='Label'):
    """Benign and attack counts, reported per split because the benign count in the
    threshold portion, not the buffer size, is what limits operating-point estimation."""
    y = df[label_col].values
    return int((y == 0).sum()), int((y == 1).sum())

def zero_event_upper(n, conf=0.95):
    """One-sided upper bound on a rate after observing no events in n trials. With two
    benign examples and no false positives the true rate is bounded only below 78%."""
    return float(1 - (1 - conf) ** (1.0 / n)) if n > 0 else float('nan')

In [ ]:
import numpy as np, pandas as pd

def reserve_then_acquire(pool, k_total, thr_frac, seed, acquire_fn=None, features=None, clean_fn=None):
    """Allocate the sampling roles before acquisition rather than after.

    A fraction thr_frac of the budget is drawn uniformly at random from the pool and reserved
    for choosing the operating threshold, so that sample is representative of target traffic.
    The remainder is acquired from the pool with those rows removed, by whatever rule is passed.
    Total purchased labels equal k_total either way, so the budget is unchanged; what changes is
    that the threshold sample is no longer drawn from the same selection as the training sample.

    Returns (fit_rows, threshold_rows)."""
    k_thr = max(1, int(round(k_total * thr_frac)))
    k_fit = max(1, k_total - k_thr)
    reserved = pool.sample(n=min(k_thr, len(pool)), random_state=seed)
    remaining = pool.drop(index=reserved.index)
    if acquire_fn is None:
        fit = remaining.sample(n=min(k_fit, len(remaining)), random_state=seed)
    else:
        X, _ = clean_fn(remaining, features)
        idx = acquire_fn(X.values, min(k_fit, len(remaining)), seed)
        fit = remaining.iloc[idx]
    return fit, reserved

def observed_fp(y_thr, p_thr, thr):
    """False positives actually observed on the threshold sample at the chosen cut, with the
    benign count, so a bound can be computed per procedure rather than from a mean sample size."""
    y = np.asarray(y_thr).astype(int); p = np.asarray(p_thr)
    benign = (y == 0)
    n = int(benign.sum())
    if n == 0 or np.isnan(thr):
        return 0, 0
    return int((p[benign] >= thr).sum()), n

def clopper_upper(k, n, conf=0.95):
    """One-sided upper confidence bound on a rate after observing k events in n trials.
    Reduces to 1 - (1-conf)^(1/n) when k = 0, and is reported to three decimals rather than
    rounded to whole percent, since a bound of 0.054% is not the same claim as 0%."""
    if n == 0:
        return float('nan')
    from scipy.stats import beta
    if k >= n:
        return 1.0
    return float(beta.ppf(conf, k + 1, n - k))

In [ ]:
from sklearn.model_selection import train_test_split

def record(row):
    pd.DataFrame([{c: row.get(c, np.nan) for c in COLS}], columns=COLS).to_csv(
        V9_CSV, mode='a', index=False, header=not os.path.exists(V9_CSV))

def key(tgt, m, b, design, rule):
    return (tgt, m, f'{float(b):.6g}', design, rule)

done = set()
if os.path.exists(V9_CSV):
    prev = pd.read_csv(V9_CSV)
    done = set(key(r.target, r.model, r.budget, r.design, r.rule) for r in prev.itertuples())
    print(f'resume: {len(done)} rows recorded')

def is_done(*k):
    return key(*k) in done

def mark(tgt, m, b, design, rule, **kw):
    record(dict(seed=CFG['seed'], target=tgt, model=m, budget=b, design=design, rule=rule, **kw))
    done.add(key(tgt, m, b, design, rule))
    print(f"  {tgt} {m} b={b} {design}/{rule}: MCC={kw.get('mcc_at_threshold', float('nan')):.3f} "
          f"FPR={kw.get('fpr_dep', float('nan')):.3f} benign_thr={kw.get('n_benign_threshold', -1)} "
          f"fp_obs={kw.get('fp_observed_threshold', -1)} bound={kw.get('fp_bound_threshold', float('nan')):.3f}")

T0 = time.time()
def el(): return f'[{(time.time()-T0)/60:5.1f}m]'

seed = CFG['seed']
parts = {}
for tag, d in DATASETS.items():
    tr, te = train_test_split(d, test_size=CFG['test_size'], stratify=d['Attack'], random_state=seed)
    tr = tr.reset_index(drop=True)
    parts[tag] = dict(train_full=tr,
                      eval=stratified_cap(te, CFG['eval_cap'], seed).reset_index(drop=True),
                      pool=stratified_cap(tr, CFG['pool_cap'], seed).reset_index(drop=True))

# the four sampling designs, built once per (target, budget). On a resume they are rebuilt only
# when something is outstanding, since the clustering is the slowest part of this notebook.
need_any = any(not is_done(tg, m, b, d, r) for tg in CFG['corpora'] for m in MODELS
               for b in CFG['budgets'] for d in ('split','reserved') for r in ('diversity','uniform'))
SAMPLES = {}
for tgt in (CFG['corpora'] if need_any else []):
    pool = parts[tgt]['pool']
    Xpool, _ = clean_X(pool, FEATURES)
    for b in CFG['budgets']:
        k = max(1, int(round(len(parts[tgt]['train_full']) * b)))
        # notebook-18 design: draw the whole buffer by one rule, then split it
        div_buf = pool.iloc[acquire_diversity(Xpool.values, min(k, len(pool)), seed)]
        uni_buf = pool.sample(n=min(k, len(pool)), random_state=seed)
        SAMPLES[(tgt, b, 'split', 'diversity')] = split_buffer(div_buf, CFG['thr_frac'], seed)
        SAMPLES[(tgt, b, 'split', 'uniform')]   = split_buffer(uni_buf, CFG['thr_frac'], seed)
        # reserved design: representative threshold sample drawn first, rest acquired from the remainder
        t0 = time.time()
        SAMPLES[(tgt, b, 'reserved', 'diversity')] = reserve_then_acquire(
            pool, k, CFG['thr_frac'], seed, acquire_fn=acquire_diversity, features=FEATURES, clean_fn=clean_X)
        SAMPLES[(tgt, b, 'reserved', 'uniform')] = reserve_then_acquire(pool, k, CFG['thr_frac'], seed)
        print(f"{el()} {tgt} b={b} k={k}: samples built ({time.time()-t0:.0f}s)")
    del Xpool; gc.collect()

for tgt in CFG['corpora']:
    ev = parts[tgt]['eval']; yev = ev['Label'].values
    pool = parts[tgt]['pool']
    benign_rate_pool = float((pool['Label'].values == 0).mean())
    for mname in MODELS:
        for b in CFG['budgets']:
            for design in ['split', 'reserved']:
                for rule in ['diversity', 'uniform']:
                    if is_done(tgt, mname, b, design, rule):
                        continue
                    fitb, thrb = SAMPLES[(tgt, b, design, rule)]
                    nb_fit, _ = label_counts(fitb); nb_thr, _ = label_counts(thrb)
                    n_labels = len(fitb) + len(thrb)
                    br_thr = float((thrb['Label'].values == 0).mean()) if len(thrb) else np.nan
                    if fitb['Label'].nunique() < 2:
                        mark(tgt, mname, b, design, rule, n_labels=n_labels, n_fit=len(fitb),
                             n_threshold=len(thrb), n_benign_fit=nb_fit, n_benign_threshold=nb_thr,
                             benign_rate_pool=benign_rate_pool, benign_rate_threshold=br_thr)
                        continue
                    Xf, med = clean_X(fitb, FEATURES)
                    Xt, _ = clean_X(thrb, FEATURES, medians=med)
                    Xe, _ = clean_X(ev, FEATURES, medians=med)
                    mdl = make_model(mname, seed, len(Xf))
                    t0 = time.time(); mdl.fit(Xf, fitb['Label'].values); fs = round(time.time()-t0, 1)
                    p_ev = mdl.predict_proba(Xe)[:, 1]
                    p_thr = mdl.predict_proba(Xt)[:, 1]
                    thr = threshold_for_fpr_on_buffer(thrb['Label'].values, p_thr, CFG['fpr_cap'])
                    fp_obs, n_ben = observed_fp(thrb['Label'].values, p_thr, thr)
                    if np.isnan(thr):
                        m_thr, td, fd = {'mcc': np.nan}, np.nan, np.nan
                    else:
                        m_thr = metrics_at_threshold(yev, p_ev, thr)
                        td, fd = realised_at_threshold(yev, p_ev, thr)
                    mark(tgt, mname, b, design, rule,
                         n_labels=n_labels, n_fit=len(fitb), n_threshold=len(thrb),
                         n_benign_fit=nb_fit, n_benign_threshold=nb_thr,
                         fp_observed_threshold=fp_obs, fp_bound_threshold=clopper_upper(fp_obs, n_ben),
                         benign_rate_pool=benign_rate_pool, benign_rate_threshold=br_thr,
                         mcc_at_threshold=m_thr['mcc'], tpr_dep=td, fpr_dep=fd,
                         tpr_oracle=tpr_at_common_fpr(yev, p_ev, CFG['fpr_cap']), thr=thr, fit_s=fs)
                    del mdl, Xf, Xt, Xe; gc.collect()

print('rows recorded:', len(done))

In [ ]:
from scipy.stats import wilcoxon

V = pd.read_csv(V9_CSV).drop_duplicates(['target','model','budget','design','rule'])

print('=== 1. DOES RESERVING A REPRESENTATIVE THRESHOLD SAMPLE FIX THE FPR MISS? ===')
t = V.groupby(['design','rule','budget']).agg(
        mcc=('mcc_at_threshold','mean'), fpr=('fpr_dep','mean'), fpr_med=('fpr_dep','median'),
        tpr=('tpr_dep','mean'), n=('mcc_at_threshold','size')).round(4)
print(t.to_string())
print('\n  target false-positive rate is 1%; realised means above are what the chosen threshold produced')

print('\n=== 2. selection bias or sample size? matched cells ===')
for rule in ['diversity','uniform']:
    for b in CFG['budgets']:
        s = V[(V.design=='split')&(V.rule==rule)&(V.budget==b)].set_index(['target','model'])
        r = V[(V.design=='reserved')&(V.rule==rule)&(V.budget==b)].set_index(['target','model'])
        j = pd.concat([s['fpr_dep'].rename('c_split'), r['fpr_dep'].rename('c_res'),
                       s['mcc_at_threshold'].rename('m_split'), r['mcc_at_threshold'].rename('m_res')],
                      axis=1).dropna()
        if len(j) < 3: continue
        p = wilcoxon(j['c_split'], j['c_res']).pvalue
        print(f"  {rule:10s} b={b}: realised FPR split {j['c_split'].mean():.4f} vs reserved {j['c_res'].mean():.4f} "
              f"(diff {(j['c_split']-j['c_res']).mean():+.4f}, p={p:.3f}, n={len(j)})")
        print(f"  {'':10s}        MCC        split {j['m_split'].mean():.3f} vs reserved {j['m_res'].mean():.3f} "
              f"(diff {(j['m_split']-j['m_res']).mean():+.3f})")

print('\n=== 3. is the threshold sample representative? ===')
# The benign-rate gap alone is not usable here: with 40 to 60 benign rows the sampling noise on
# that rate is 0.03 to 0.06, which swamps a moderate selection effect. The powered signal is the
# benign count against what a uniform draw would give, since coverage selection enriches rare
# classes by construction.
V['benign_expected'] = V['n_threshold'] * V['benign_rate_pool']
V['benign_ratio'] = V['n_benign_threshold'] / V['benign_expected'].replace(0, np.nan)
rep = V.groupby(['design','rule']).agg(
        benign_obs=('n_benign_threshold','mean'), benign_exp=('benign_expected','mean'),
        ratio=('benign_ratio','mean')).round(3)
print(rep.to_string())
print('\n  a reserved sample is a uniform draw, so its observed-over-expected ratio should sit at 1;')
print('  a split coverage-selected sample enriches rare benign traffic and should exceed it.')
print('\n  ratio by target, largest where benign traffic is rarest:')
print(V.groupby(['target','design','rule'])['benign_ratio'].mean().round(2).to_string())

print('\n=== 4. PER-PROCEDURE BOUNDS, not a mean-sample-size illustration ===')
b_ = V[V['fp_bound_threshold'].notna()]
print(b_.groupby(['target','budget','design']).agg(
        benign=('n_benign_threshold','mean'), fp_obs=('fp_observed_threshold','mean'),
        bound=('fp_bound_threshold','mean'), realised=('fpr_dep','mean')).round(4).to_string())
print('\n  bound is Clopper-Pearson at 95% from the counts of each individual procedure;')
print('  where the realised FPR exceeds the bound, the threshold sample did not represent deployment traffic.')
over = b_[b_['fpr_dep'] > b_['fp_bound_threshold']]
print(f"  procedures whose realised FPR exceeds their own bound: {len(over)} of {len(b_)}")
if len(over):
    print(over.groupby(['design','rule']).size().to_string())
V.round(6).to_csv(f'{RESULT}/fc_reserved_threshold.csv', index=False)
print('\nsaved fc_reserved_threshold.csv')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
os.chdir('/content/drive/MyDrive/drift-conference')

r = subprocess.run(["python", "tools/commit_cell.py",
  "19: reserved representative threshold sample, selection bias vs sample size, per-procedure bounds"],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)